# Lab: Spark in Practice

This lab runs inside a Docker container. The image already includes Python, Java, PySpark, and Jupyter, so the only thing you need to install on your machine is Docker Desktop.

**How to use this notebook:** run the cells in order, one at a time (`Shift+Enter`). Between missions, refresh the Spark UI at [http://localhost:4040](http://localhost:4040). Each mission indicates what you should see in the interface; if you see something else, stop and investigate why.

**Expected numbers:**

| What | Value |
|---|---|
| csv.gz when reading it | 1 partition (gzip is not splittable) |
| `data/travel` when reading it | 4 partitions |
| narrow (filter+select+collect) | 1 stage, 4 tasks |
| wide (groupBy+collect) | 2 stages, 12 tasks |
| 2 wide (groupBy+orderBy) | 3 stages, 20 tasks |
| cache after an action | Storage: 4 partitions, ~28 MB |

Keep in mind that the Spark UI only exists while the kernel has a live Spark session. If you restart the kernel, you need to run again from Mission 3.

### Concept covered in each mission

| Mission | Concept | 
|---|---|
| 1–2 | Partition · splittable formats
| 4 | 1 partition = 1 task · `count()` generates 2 stages
| 5 | Lazy evaluation · transformation vs action
| 6 | Wide → shuffle · `shuffle.partitions` 
| 7 | Stage = shuffles + 1 
| 8 | Storage memory · `cache()` 
| 9 | Physical plan · `Exchange` = shuffle


## Mission 1 — Download the dataset *(once)*

Dataset: 1.37 million New York taxi trips (January 2021). This is open city data published on GitHub by DataTalksClub. It is ~25 MB and does not require registration.

Because the folder is mounted into the container, the file is saved on your machine and will not be downloaded again even if you stop the container.


In [1]:
import os, urllib.request

URL = ("https://github.com/DataTalksClub/nyc-tlc-data/releases/download/"
       "yellow/yellow_tripdata_2021-01.csv.gz")
FILE = "yellow_tripdata_2021-01.csv.gz"

if not os.path.exists(FILE):
    print("Downloading ~25 MB...")
    urllib.request.urlretrieve(URL, FILE)
    print("Downloaded:", os.path.getsize(FILE) // 1_000_000, "MB")
else:
    print("The dataset is already downloaded.")


Downloaded: 25 MB


## Mission 2 — Prepare the dataset *(once)*

We convert the csv.gz into 4 parquet files and rename the columns in Spanish. Keep two things in mind:

1. A gzip file is not splittable. No matter how large it is, Spark can only read it with one task. You can see this in the partition count.
2. `repartition(4)` + `write` generates 4 parquet files. This lets us control how many partitions the lab will have.


In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

if not os.path.exists("data/travel"):

    prep = (
        SparkSession.builder
        .appName("preparation")
        .master("local[4]")
        .getOrCreate()
    )

    prep.sparkContext.setLogLevel("ERROR")

    raw = prep.read.csv(FILE, header=True, inferSchema=True)

    print("csv.gz partitions:", raw.rdd.getNumPartitions())

    travel = (
        raw.select(
            F.col("PULocationID").alias("zone"),
            F.col("payment_type").alias("payment_type"),
            F.col("passenger_count").alias("passengers"),
            F.col("trip_distance").alias("distance"),
            F.col("total_amount").alias("total")
        )
        .filter(F.col("total") > 0)
    )

    travel.repartition(4).write.mode("overwrite").parquet("data/travel")

    print("Done: data/travel with 4 parquet files.")

    prep.stop()

else:
    print("data/travel already exists.")

csv.gz partitions: 1


Py4JJavaError: An error occurred while calling o52.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:793)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1249)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1454)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:601)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:761)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.getAllCommittedTaskPaths(Fil

## Mission 3 — The lab session

Three configurations to keep in mind:

- **AQE disabled.** With AQE, stages and tasks are adjusted at runtime and the numbers are no longer predictable. In the next session, we will enable it to see what changes. In production, it should not be disabled.
- **shuffle.partitions = 8.** So that the counts are easy to read. The default value is 200 (exam fact).
- **maxPartitionBytes = 8m.** By default, Spark combines small files into a single partition. With this setting, each parquet file becomes one partition and everyone sees the same numbers.

After running this cell, open the Spark UI at [http://localhost:4040](http://localhost:4040), **Jobs** tab.


In [ ]:
spark = (
    SparkSession.builder
    .appName("live30b-spark-en-la-practica")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.files.maxPartitionBytes", "8m")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Session ready. Spark UI -> http://localhost:4040")


## Mission 4 — 1 partition = 1 task, and `count()` generates 2 stages

**You should see:** `partitions: 4` and ~1.36M rows. Two jobs appear in the UI:

| Job | What it is | Stages | Tasks |
|---|---|---|---|
| 0 · `parquet` | Spark reading the file schemas. This is not your query | 1 | 1 |
| 1 · `count` | The count | 2 | 5 |

Before looking at the UI: how many stages does a `count()` have?

There are two, with 5 tasks = 4 + 1:

- Stage 1, 4 tasks: each task counts the rows in its partition. One partition, one task.
- Stage 2, 1 task: it adds the four counts together. It needs data from multiple partitions, meaning it is a wide operation, which requires a shuffle and therefore a new stage.

That shuffle moved 236 bytes (four numbers) and still split the job into two stages. A shuffle depends on the operation, not on the volume of data.

Open job 1 and its two stages to see it.


In [ ]:
df = spark.read.parquet("data/travel")
print("partitions:", df.rdd.getNumPartitions())   # SHOULD BE: 4
df.printSchema()
print("rows:", df.count())


## Mission 5 — Lazy evaluation

First cell: three transformations. **You should see in the UI:** the number of jobs has not changed. Spark has only recorded the plan.


In [ ]:
t = (df
     .filter(F.col("total") > 50)                 # expensive trips
     .select("zona", "total")
     .withColumn("propina_sugerida", F.col("total") * 0.10))

print("Transformations applied. Refresh the UI: the number of jobs does not change.")


Second cell: the action. **You should see:** a new job, with 1 stage and 4 tasks.

Four partitions, four tasks. Filter, select, and withColumn are narrow transformations: each task works only with its own partition, so there is no shuffle and no new stage.


In [ ]:
filas = t.collect()
print("travel de más de $50:", len(filas))        # ~45,000


## Mission 6 — Wide: the shuffle appears

**You should see:** the new job has 2 stages and 12 tasks.

12 = 4 (read) + 8 (`shuffle.partitions`). The shuffle determines how many partitions exist after it, and therefore how many tasks.


In [ ]:
ingresos = df.groupBy("zona").agg(F.sum("total").alias("ingresos")).collect()
print("zonas:", len(ingresos))                    # 258


## Mission 7 — Two wide operations: 3 stages

**You should see:** 3 stages, 20 tasks. groupBy and orderBy are two shuffles; the rule is stages = shuffles + 1.

The zone with the highest revenue is 132, which corresponds to JFK Airport.


In [ ]:
(df.groupBy("zone")
   .agg(F.sum("total").alias("revenue"))
   .orderBy(F.desc("revenue"))
   .show(5))

## Mission 8 — Storage memory: `cache()` is also lazy

**New concept.** That memory is divided into two areas:

```
EXECUTOR MEMORY
├── execution memory  ── joins, sorts, aggregations  (when it does not fit → spill, next session)
└── storage memory    ── DataFrame cache         (the one seen in this mission)
```

Run the first cell and go to the **Storage** tab in the UI. **You should see:** empty. `cache()` is a transformation and, like every transformation, it is lazy.


In [ ]:
df.cache()
print("Check the Storage tab: empty.")

Now the action. **You should see in Storage:** the DataFrame, 4 partitions, ~28 MB. Without an action, there is no cache.


In [ ]:
df.filter(F.col("total") > 0).select("zone").collect()
print("Refresh the Storage tab: 4 partitions in memory.")

## Mission 9 — The physical plan: SQL and DataFrame API generate the same plan

**New concept.** Spark optimizes the plan before execution (for example, column pruning). This optimizer is called Catalyst, and the plan can be inspected with `explain()`.

**You should see:** the same physical plan in both cases. Using SQL or the DataFrame API is a matter of preference; underneath, they use the same plan.

Two elements of the plan to identify:

- `Exchange hashpartitioning(zona, 8)`: the shuffle, using the configured 8 `shuffle.partitions`.
- `InMemoryTableScan`: it is reading from the cache created in Mission 8, not from the parquet files.


In [ ]:
df.createOrReplaceTempView("travel")

print("--- SQL plan ---")
spark.sql("SELECT zone, SUM(total) AS income FROM travel GROUP BY zone").explain()

print("--- DataFrame API plan ---")
df.groupBy("zone").agg(F.sum("total").alias("revenue")).explain()


## Exercises

1. **`count()`.** `df.count()` was already executed in Mission 4. Find it in the UI: how many stages did it have? Why is it not 1? *(Hint: count per partition and then combine the counts.)*
2. **The default value.** Change `shuffle.partitions` to `"200"` (restart the kernel and run from Mission 3) and repeat Mission 6. How many tasks? Did the runtime improve or get worse with 1.36M rows?
3. **Narrow or wide.** Without running them, classify: `dropDuplicates()`, `limit(10)`, `repartition(2)`, `withColumnRenamed()`. Then run each with `.collect()` and verify against the stages.
4. **Cache with measurement.** Repeat Mission 6 twice with cached `df`, measuring with `time.time()`. Is the second run faster? Which memory is it reading from?
5. **Partitions.** Remove the `maxPartitionBytes` configuration from Mission 3, restart, and repeat Mission 4. How many partitions are there now? Why?
6. **Business question.** What percentage of trips are paid by card (`tipo_pago = 1`)? Write the query, predict the stages before running it, and verify.

When finished: run `spark.stop()` in a cell, or shut down the container with `Ctrl+C` / `docker compose down`.
